# Дообучение реранкера для телеграм бота

ноутбук с экспериментами с дообучением `BAAI/bge-reranker-v2-m3` на подготовленном даатсете с мемами

## подготовка среды


In [1]:
!nvidia-smi || true

Fri Apr  3 09:42:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


у меня есть два файла: `memes_retrieval.csv` и `reranker_pairs.csv`. первая - это таблица с мемами (пути до них) и текстовыми полями, дающие им характеристику

вторая - таблица с обучающими парами (позитивными и негативными мемами под сообщение пользователя) для реранкера.

In [3]:
from pathlib import Path

meme_dataset_path = Path('/content/drive/MyDrive/dataset/memes_retrieval.csv')
pairs_dataset_path = Path('/content/drive/MyDrive/dataset/reranker_pairs.csv')

# результаты будем сохранять в отдельную папку
output_root = Path('/content/drive/MyDrive/dataset/reranker_experiments')
output_root.mkdir(parents=True, exist_ok=True)

baai_model = 'BAAI/bge-reranker-v2-m3'

# зададим текстовые наборы полей, чтобы сравнить, на каком реранкер будет работать лучше
text_fields_config = {
  'full': ['embedding_text', 'ocr_text', 'semantic_description'],
  'embedding_only': ['embedding_text'],
  'embedding_plus_ocr': ['embedding_text', 'ocr_text'],
}

In [4]:
!pip install sentence-transformers==2.3.1 transformers==4.38.2 huggingface_hub==0.23.0 openpyxl scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing ins

In [5]:
import random
from sentence_transformers import CrossEncoder, InputExample
import pandas as pd
import json
from collections import Counter
from dataclasses import dataclass
from torch.utils.data import DataLoader

random.seed(42)

## подготовка данных

построим dataclass-структуры для обучающих примеров и для оценки ранжирорвания от реранкера

In [6]:
@dataclass(frozen=True)
class PairExample:
  query: str
  candidate_text: str
  label: int
  positive_meme_id: str
  candidate_meme_id: str
  source: str


@dataclass(frozen=True)
class RankingGroup:
  query: str
  positive_meme_id: str
  candidate_meme_ids: list[str]
  candidate_texts: list[str]
  labels: list[int]
  source: str

In [7]:
def load_dataframe(path: Path) -> pd.DataFrame:
  return pd.read_csv(path, encoding='cp1251', sep=';')

нормализуем текст, функция возвращает просто строку без лишних отступов

In [8]:
def normalize_text(value) -> str:
  if value is None:
    return ''
  text = str(value).strip()
  if text.lower() == 'nan':
    return ''
  return text

парсим hard_negative примеры

In [9]:
def parse_hard_negative_memes(value) -> list[str]:
  text = normalize_text(value)
  if not text:
    return []
  try:
    parsed = json.loads(text)
  except json.JSONDecodeError:
    return []
  if not isinstance(parsed, list):
    return []
  return [str(item).strip() for item in parsed if str(item).strip()]

In [10]:
def build_meme_text_mapping(df: pd.DataFrame, text_columns: list[str]) -> dict[str, str]:
  mapping = {}
  for _, row in df.iterrows():
    meme_id = normalize_text(row.get('meme_id', ''))
    if not meme_id:
      continue
    parts = []
    for column in text_columns:
      value = normalize_text(row.get(column, ''))
      if value:
        parts.append(f'{column}: {value}')
    candidate_text = ' | '.join(parts)
    if candidate_text:
      mapping[meme_id] = candidate_text
  return mapping

функция, которая строит обьучающие примеры для CrossEncoder'а

In [11]:
def build_pair_examples(pair_df: pd.DataFrame, meme_text_by_id: dict[str, str]) -> list[PairExample]:
    examples = []
    for _, row in pair_df.iterrows():
      query = normalize_text(row.get('query', ''))
      positive_meme_id = normalize_text(row.get('positive_meme_id', ''))
      source = normalize_text(row.get('source', '')) or 'unknown'
      negative_ids = parse_hard_negative_memes(row.get('hard_negative_ids', ''))
      if not query or not positive_meme_id or positive_meme_id not in meme_text_by_id:
        continue

      examples.append(
          PairExample(
              query=query,
              candidate_text=meme_text_by_id[positive_meme_id],
              label=1,
              positive_meme_id=positive_meme_id,
              candidate_meme_id=positive_meme_id,
              source=source,
          )
      )

      for negative_meme_id in negative_ids:
        candidate_text = meme_text_by_id.get(negative_meme_id)
        if not candidate_text or negative_meme_id == positive_meme_id:
          continue
        examples.append(
          PairExample(
            query=query,
            candidate_text=candidate_text,
            label=0,
            positive_meme_id=positive_meme_id,
            candidate_meme_id=negative_meme_id,
            source=source,
          )
        )
    return examples

для каждого запроса собитраем группу с 1 положительным мемом и остальными отрицательными, потом модель выдает метрику и показывает, на каком месте стоит наиболее правильный мем

In [12]:
def build_ranking_groups(pair_df: pd.DataFrame, meme_text_by_id: dict[str, str]) -> list[RankingGroup]:
    groups = []
    for _, row in pair_df.iterrows():
      query = normalize_text(row.get('query', ''))
      positive_meme_id = normalize_text(row.get('positive_meme_id', ''))
      source = normalize_text(row.get('source', '')) or 'unknown'
      negative_ids = parse_hard_negative_memes(row.get('hard_negative_ids', ''))
      positive_text = meme_text_by_id.get(positive_meme_id)
      if not query or not positive_text:
        continue

      candidate_meme_ids = [positive_meme_id]
      candidate_texts = [positive_text]
      labels = [1]

      for negative_meme_id in negative_ids:
        negative_text = meme_text_by_id.get(negative_meme_id)
        if not negative_text or negative_meme_id == positive_meme_id:
          continue
        candidate_meme_ids.append(negative_meme_id)
        candidate_texts.append(negative_text)
        labels.append(0)

      if len(candidate_meme_ids) < 2:
        continue

      groups.append(
        RankingGroup(
          query=query,
          positive_meme_id=positive_meme_id,
          candidate_meme_ids=candidate_meme_ids,
          candidate_texts=candidate_texts,
          labels=labels,
          source=source,
        )
      )
    return groups


In [13]:
def split_by_meme_id(pair_df: pd.DataFrame, val_ratio: float, seed: int):
  meme_ids = sorted({normalize_text(value) for value in pair_df['positive_meme_id'].tolist() if normalize_text(value)})
  rng = random.Random(seed)
  rng.shuffle(meme_ids)
  val_count = max(1, int(len(meme_ids) * val_ratio))
  val_ids = set(meme_ids[:val_count])
  train_ids = set(meme_ids[val_count:])
  return train_ids, val_ids


def filter_pair_df_by_positive_ids(pair_df: pd.DataFrame, positive_ids: set[str]) -> pd.DataFrame:
  mask = pair_df['positive_meme_id'].astype(str).str.strip().isin(positive_ids)
  return pair_df[mask].copy()

обертка, возвращающая все сразу (словарь, примеры для обучения, RankingGroups)

In [14]:
def build_examples_and_groups(meme_df: pd.DataFrame, pair_df: pd.DataFrame, text_columns: list[str]):
  meme_text_by_id = build_meme_text_mapping(meme_df, text_columns)
  examples = build_pair_examples(pair_df, meme_text_by_id)
  groups = build_ranking_groups(pair_df, meme_text_by_id)
  return meme_text_by_id, examples, groups

## загрузка датасетов

In [15]:
meme_df = load_dataframe(meme_dataset_path)
pair_df = load_dataframe(pairs_dataset_path)

print('meme_df shape:', meme_df.shape)
print('pair_df shape:', pair_df.shape)

meme_df shape: (150, 14)
pair_df shape: (3134, 6)


In [16]:
display(meme_df.head(3))
display(pair_df.head(5))

,meme_id,image_path,file_name,split,ocr_text,semantic_description,emotions,situations,humor_style,user_messages,search_queries,query_examples,safety_notes,embedding_text
0,meme_001,C:\Users\Lenovo\Desktop\dataset\train\images\m...,meme_001.jpg,train,ТГ: ШИЗЕЮ объебись,Мем выражает состояние сильного эмоционального...,"[""усталость"", ""шок"", ""перегрузка"", ""раздражени...","[""когда слишком много работы"", ""после долгого ...","[""ирония"", ""сарказм"", ""самоирония""]","[""Я просто не могу больше так работать."", ""Это...","[""усталость и перегрузка"", ""эмоциональное выго...","[""Я просто не могу больше так работать."", ""Это...",NaN,Мем выражает сильное эмоциональное перенапряже...
1,meme_002,C:\Users\Lenovo\Desktop\dataset\train\images\m...,meme_002.jpg,train,НАДОЕЛО ТИХО ХОДИТЬ,Мем выражает усталость от необходимости вести ...,"[""раздражение"", ""усталость"", ""фрустрация"", ""ре...","[""когда надоело скрывать свои мысли"", ""когда у...","[""ирония"", ""сарказм"", ""гипербола""]","[""Ну всё, хватит терпеть!"", ""Я больше не могу ...","[""усталость от сдержанности"", ""раздражение и ж...","[""Ну всё, хватит терпеть!"", ""Я больше не могу ...",NaN,Мем выражает усталость и раздражение от необхо...
2,meme_003,C:\Users\Lenovo\Desktop\dataset\train\images\m...,meme_003.jpg,train,МУСЯ ЭТО ТЫ О ЭТО МУСЯ С МУСЯТАМИ,Мем выражает забавное сравнение между одним че...,"[""юмор"", ""ирония"", ""легкость"", ""забава""]","[""когда кто-то говорит о себе и своих друзьях""...","[""ирония"", ""легкий сарказм"", ""игра слов""]","[""Ну ты и компания у себя!"", ""Вот это семейка ...","[""шутка про сходство с группой"", ""ирония про о...","[""Ну ты и компания у себя!"", ""Вот это семейка ...",NaN,Мем с текстом 'МУСЯ ЭТО ТЫ О ЭТО МУСЯ С МУСЯТА...


,target_meme_id,query,positive_meme_id,hard_negative_ids,source,hardness_reason
0,meme_001,"Я вообще не знаю как я еще работаю, уже схожу ...",meme_001,[],synthetic_positive_query,NaN
1,meme_001,После этого дня мне хочется просто убежать куд...,meme_001,[],synthetic_positive_query,NaN
2,meme_001,"Это настолько абсурдно, что я не могу, у меня ...",meme_001,[],synthetic_positive_query,NaN
3,meme_001,"Мозг кипит, кажется, щас взорвусь",meme_001,[],synthetic_positive_query,NaN
4,meme_001,"Начальник в шесть присылает кучу задач, я на г...",meme_001,[],synthetic_positive_query,NaN


делим мемы на train и val

In [17]:
train_ids, val_ids = split_by_meme_id(pair_df, val_ratio=0.22, seed=42)
train_pair_df = filter_pair_df_by_positive_ids(pair_df, train_ids)
val_pair_df = filter_pair_df_by_positive_ids(pair_df, val_ids)

print('train meme count:', len(train_ids))
print('val meme count:', len(val_ids))

# проверка на data leaks
print('intersection:', len(train_ids & val_ids))

train meme count: 116
val meme count: 32
intersection: 0


проверяем кол-во запросов на каждый мем в обоих выборках, чтобы убедиться, нет ли перекосов

In [18]:
train_counter = Counter(train_pair_df['positive_meme_id'].astype(str))
val_counter = Counter(val_pair_df['positive_meme_id'].astype(str))

print('train avg queries per meme:', round(sum(train_counter.values()) / max(len(train_counter), 1), 2))
print('val avg queries per meme:', round(sum(val_counter.values()) / max(len(val_counter), 1), 2))

train avg queries per meme: 21.16
val avg queries per meme: 21.25


## Эксперименты

In [ ]:
def evaluate_group_ranking(model: CrossEncoder, groups: list[RankingGroup]) -> dict:
  recall_at_1 = 0
  recall_at_3 = 0
  reciprocal_rank_sum = 0.0
  failures = []

  for group in groups:
    pairs = [[group.query, text] for text in group.candidate_texts]
    scores = model.predict(pairs)
    ranked = sorted(
      zip(group.candidate_meme_ids, group.labels, scores),
      key=lambda item: item[2],
      reverse=True,
    )

    rank = None
    for idx, (_, label, _) in enumerate(ranked, start=1):
      if label == 1:
        rank = idx
        break

    if rank == 1:
      recall_at_1 += 1
    if rank is not None and rank <= 3:
      recall_at_3 += 1
    if rank is not None:
      reciprocal_rank_sum += 1.0 / rank
    if rank != 1:
      failures.append(
        {
          'query': group.query,
          'positive_meme_id': group.positive_meme_id,
          'rank': rank,
          'ranked_ids': [item[0] for item in ranked],
          'scores': [float(item[2]) for item in ranked],
          'source': group.source,
        }
      )

  total = max(len(groups), 1)
  return {
    'count': len(groups),
    'recall@1': recall_at_1 / total,
    'recall@3': recall_at_3 / total,
    'mrr': reciprocal_rank_sum / total,
    'failures': failures,
  }


def print_metrics(title: str, metrics: dict):
  print(title)
  print(f"count: {metrics['count']}")
  print(f"recall@1: {metrics['recall@1']:.4f}")
  print(f"recall@3: {metrics['recall@3']:.4f}")
  print(f"mrr: {metrics['mrr']:.4f}")

baseline по разным текстовым конфигурациям

In [20]:
baseline_results = {}

for config_name, text_columns in text_fields_config.items():
  # здесь собираем группы кандидатов для валидации
  _, _, val_groups = build_examples_and_groups(meme_df, val_pair_df, text_columns)
  model = CrossEncoder(baai_model, num_labels=1, max_length=512)

  metrics = evaluate_group_ranking(model, val_groups)
  baseline_results[config_name] = metrics
  print_metrics(f'baseline / {config_name}', metrics)
  print('-' * 60)

baseline_table = pd.DataFrame(
  [
    {
      'config': config_name,
      'count': metrics['count'],
      'recall@1': metrics['recall@1'],
      'recall@3': metrics['recall@3'],
      'mrr': metrics['mrr'],
    }
    for config_name, metrics in baseline_results.items()
  ]
).sort_values(['recall@1', 'mrr'], ascending=False)

display(baseline_table)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

baseline / full
count: 358
recall@1: 0.8575
recall@3: 0.9972
mrr: 0.9230
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


baseline / embedding_only
count: 358
recall@1: 0.8603
recall@3: 0.9972
mrr: 0.9234
------------------------------------------------------------
baseline / embedding_plus_ocr
count: 358
recall@1: 0.8687
recall@3: 0.9972
mrr: 0.9281
------------------------------------------------------------


,config,count,recall@1,recall@3,mrr
2,embedding_plus_ocr,358,0.868715,0.997207,0.928073
1,embedding_only,358,0.860335,0.997207,0.923417
0,full,358,0.857542,0.997207,0.922952


видим, что embedding_plus_ocr показывает себя лучше других сочетаний, поэтому будем дообучать модель, используя эти колонки

In [ ]:
primary_text_columns = text_fields_config['embedding_plus_ocr']
_, train_examples, train_groups = build_examples_and_groups(meme_df, train_pair_df, primary_text_columns)
_, val_examples, val_groups = build_examples_and_groups(meme_df, val_pair_df, primary_text_columns)

train_dataloader = DataLoader(
  [InputExample(texts=[item.query, item.candidate_text], label=float(item.label)) for item in train_examples],
  batch_size=8,
  shuffle=True,
)

finetuned_dir = output_root / 'finetuned_embedding_plus_ocr'
finetuned_dir.mkdir(parents=True, exist_ok=True)

model = CrossEncoder(baai_model, num_labels=1, max_length=512)
warmup_steps = max(1, int(len(train_dataloader) * 2 * 0.1))

model.fit(
  train_dataloader=train_dataloader,
  epochs=2,
  warmup_steps=warmup_steps,
  optimizer_params={'lr': 2e-5},
  show_progress_bar=True,
)

model.save(str(finetuned_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Iteration:   0%|          | 0/644 [00:00<?, ?it/s]

Iteration:   0%|          | 0/644 [00:00<?, ?it/s]

In [22]:
baseline_model = CrossEncoder(baai_model, num_labels=1, max_length=512)
finetuned_model = CrossEncoder(str(finetuned_dir), num_labels=1, max_length=512)

baseline_metrics = evaluate_group_ranking(baseline_model, val_groups)
finetuned_metrics = evaluate_group_ranking(finetuned_model, val_groups)

comparison_df = pd.DataFrame([
  {
    'model': 'baseline',
    'count': baseline_metrics['count'],
    'recall@1': baseline_metrics['recall@1'],
    'recall@3': baseline_metrics['recall@3'],
    'mrr': baseline_metrics['mrr'],
  },
  {
    'model': 'finetuned',
    'count': finetuned_metrics['count'],
    'recall@1': finetuned_metrics['recall@1'],
    'recall@3': finetuned_metrics['recall@3'],
    'mrr': finetuned_metrics['mrr'],
  },
])

display(comparison_df)

,model,count,recall@1,recall@3,mrr
0,baseline,358,0.868715,0.997207,0.928073
1,finetuned,358,0.709497,0.997207,0.839153


посмотрим на ошибки

In [24]:
failure_df = pd.DataFrame(finetuned_metrics['failures'])
print('number of failures:', len(failure_df))
display(failure_df.head(50))

number of failures: 104


,query,positive_meme_id,rank,ranked_ids,scores,source
0,Когда кто-то делает тупую вещь и ты пытаешься ...,meme_032,2,"[meme_085, meme_032, meme_125]","[0.0007646773592568934, 0.0002611058880575001,...",synthetic_hard_negative
1,Обсуждение в чате превращается в набор абсурда...,meme_032,2,"[meme_085, meme_032, meme_097]","[0.0003072441031690687, 0.0002198757865699008,...",synthetic_hard_negative
2,"Иногда мои мысли такие нелогичные, что кажется...",meme_032,2,"[meme_082, meme_032, meme_148]","[0.9966720342636108, 0.11716129630804062, 0.00...",synthetic_hard_negative
3,"Пытаюсь объяснить что-то простое, а все продол...",meme_032,2,"[meme_004, meme_032, meme_002]","[0.00011379550414858386, 7.232093048514798e-05...",synthetic_hard_negative
4,Когда окружающие ведут себя нелогично и единст...,meme_032,2,"[meme_085, meme_032, meme_125]","[0.9548238515853882, 0.0002458471863064915, 4....",synthetic_hard_negative
5,"Почему я до сих пор злюсь на него, хотя прошло...",meme_034,3,"[meme_033, meme_141, meme_034]","[0.00012498520663939416, 5.409902587416582e-05...",synthetic_hard_negative
6,"Он извинился, но мне как-то не верится — я всё...",meme_034,2,"[meme_033, meme_034, meme_042]","[0.9999772310256958, 0.9988691210746765, 3.709...",synthetic_hard_negative
7,"Хочу как-то саркастично дать понять, что помню...",meme_034,2,"[meme_033, meme_034, meme_054]","[0.999326229095459, 0.9929379820823669, 0.0001...",synthetic_hard_negative
8,"Не могу простить человека, хотя он извиняется ...",meme_034,2,"[meme_033, meme_034, meme_146]","[0.9999244213104248, 5.059292016085237e-05, 4....",synthetic_hard_negative
9,"Хочу подколоть человека, который постоянно пов...",meme_034,3,"[meme_144, meme_141, meme_034]","[0.0003089471720159054, 7.146799907786772e-05,...",synthetic_hard_negative


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


сохраним модель н гугл диск

In [39]:
# del baseline_model
del finetuned_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
primary_text_columns = text_fields_config['embedding_plus_ocr']
_, train_examples, train_groups = build_examples_and_groups(meme_df, train_pair_df, primary_text_columns)
_, val_examples, val_groups = build_examples_and_groups(meme_df, val_pair_df, primary_text_columns)

train_dataloader = DataLoader(
  [InputExample(texts=[item.query, item.candidate_text], label=float(item.label)) for item in train_examples],
  batch_size=8,
  shuffle=True,
)

finetuned_2_dir = output_root / 'finetuned_embedding_plus_ocr_2'
finetuned_2_dir.mkdir(parents=True, exist_ok=True)

model = CrossEncoder(baai_model, num_labels=1, max_length=512)
warmup_steps = max(1, int(len(train_dataloader) * 1 * 0.1))

model.fit(
  train_dataloader=train_dataloader,
  epochs=1,
  warmup_steps=warmup_steps,
  optimizer_params={'lr': 2e-5},
  show_progress_bar=True,
)

model.save(str(finetuned_2_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/644 [00:00<?, ?it/s]

In [41]:
finetuned_2_model = CrossEncoder(str(finetuned_2_dir), num_labels=1, max_length=512)
finetuned_2_metrics = evaluate_group_ranking(finetuned_2_model, val_groups)

comparison_df = pd.DataFrame([
  {
    'model': 'finetuned',
    'count': finetuned_2_metrics['count'],
    'recall@1': finetuned_2_metrics['recall@1'],
    'recall@3': finetuned_2_metrics['recall@3'],
    'mrr': finetuned_2_metrics['mrr'],
  },
])

display(comparison_df)

,model,count,recall@1,recall@3,mrr
0,finetuned,358,0.768156,0.997207,0.869413


In [42]:
from google.colab import files
import shutil

shutil.make_archive('finetuned_model_2', 'zip', finetuned_2_dir)

files.download('finetuned_model_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>